# Clinical EDA
Exploring the Data for the first Datasheet regarding all Clinical Figshares, it only serves as a first look to get familiar with the structure and some values

# Setup & Imports

In [ ]:


print("Loading Libraries")
# Core Libraries

import pandas as pd # excel tools
import numpy as np # math
print("Done loading")

Loading Libraries
Done loading


# Load Data from Excel Sheet
Loading from the file and organising it through its sheets

In [2]:
#reading from excel file
all_sheets = pd.read_excel('../datasets/RA_MAP_Clinical_Figshare_17_5_21.xlsx', sheet_name=None)
#extracting and renaming sheet names
print(all_sheets.keys())
df_clinical = all_sheets['OpenPseudonymised_RA_MAP_Clinic']
df_steroids = all_sheets['intramuscular steroids']
df_meds = all_sheets['RA Meds']
df_glossary = all_sheets['Glossary']
#columns and rows per sheet
for name, sheet in all_sheets.items():
    print(f'{name}: {sheet.shape[0]} rows x {sheet.shape[1]} columns')


FileNotFoundError: [Errno 2] No such file or directory: '../datasets/RA_MAP_Clinical_Figshare_17_5_21.xlsx'

---
## Total Amount of Cells excluding glossary

df_clinical = 21909,
df_steroids = 4140,
df_meds = 22806,
total = 48855


---
## First look at Data
Here we take a look at the structure of the Data (cell/column names)

In [ ]:
# Styling option to show whole table
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
# whole Glossary to get familiar with terms
df_glossary

In [ ]:
# first, last and random five of regular clinical metadata 
display(df_clinical.head())
display(df_clinical.tail())
display(df_clinical.sample(5))

# Deeper Analysis
Now we take a patient from the previous cell and walk through the sheets
we can also do this with a random patient. Just comment it in
to do this we save the id and digest as a variable

In [ ]:
#custom query to filter by Patient_ID, saving as patient and patient_id to use for further filtering
#patient = df_clinical[df_clinical['Patient_ID'] == 'TAC1001']
patient = df_clinical.sample() # random patient
patient_id = patient['Patient_ID'].item()
display(patient.T)
display(patient_id)

In [ ]:
# exemplary columns and saving the digest for further showcasing
columns_needed = ["Digest", "Study", "WEIGHT", "GENDER","DAS28.0M","DAS28.9M", "RHUEMATOID.FACTOR", "CURENT SMOKER"]
display(patient[columns_needed].T)
# .item() to save the string literal
patient_digest = patient["Digest"].item()
print(f"Patient {patient_id} digest is: {patient_digest}")

# Filter
The digest is the main identifier across all Sheets, some also used the Patient_ID with the timestamp e.g. "TAC1932_6M", "TAC1000_BL"
For this we have more scripts now

## Sheet Appearances
First we check the appearance of a patient by their respective digest with the find_patient() function

In [ ]:
#method to call upon to check where they show up, uses the patient_id as input
#since we created a patient_id variable we can either call it or with whatever patient_id string we use as input
def find_patient(patient_id):
    digest = df_clinical[df_clinical['Patient_ID'] == str(patient_id)]['Digest']
    #check if patient exists
    if digest.empty:
        print(f'{patient_id}: Patient does not exist')
        return
    
    #extracts literal string and checks if it shows up in respective sheets
    digest = digest.item()
    in_steroids = digest in df_steroids['Digest'].values
    in_meds = digest in df_meds['Digest'].values
    
    print(f'Does {patient_id} show up?')
    print(f'Clinical:  Yes') #hardcoded since they exist by definition
    print(f'Steroids:  {"Yes" if in_steroids else "No"}')
    print(f'Meds:      {"Yes" if in_meds else "No"}')
    print(f'Across all:  {"Yes" if (in_steroids and in_meds) else "No"}')

find_patient(patient_id)

In [ ]:
# script to show all patients that show up across all sheets
#indexing, uses drop_duplicates() since digests are used more than once in df_steroids and df_meds
common_patients = df_clinical[
    df_clinical['Digest'].isin(df_steroids['Digest']) &
    df_clinical['Digest'].isin(df_meds['Digest'])
][['Patient_ID', 'Digest']].drop_duplicates()
#show
print(f'Patients in all three sheets: {len(common_patients)}')
display(common_patients)

# Steroid Injections Dataframe
According to [this Blog](https://www.rheumatoidarthritis.org/treatment/medications/corticosteroids/index.html). Steroids are used to reduce inflammation and help in autoimmune activity regulation. It is also often used as a "bridge therapy", i. e. a sort of boost while waiting for the medications to begin working
It is important to note that some patients did not receive an injection. Thus there are a lot of checks happening at this section of the script. If they did receive one, then it often happened at the same time they started their therapy. This is accounted for in the "overlap" section.
 

In [ ]:
# first, last and random five of steroid sheet
display(df_steroids.head())
display(df_steroids.tail())
display(df_steroids.sample(5))

In [ ]:
steroid_patient = df_steroids[df_steroids['Digest'] == patient_digest]
timestamps = steroid_patient['Assessment'].drop_duplicates().sort_values().values
if steroid_patient.empty:
    print('Patient did not get a steroid dose')
else:
    display(steroid_patient)
    print(f'Total amount of injections: {len(steroid_patient)}')
    print(f'The patient {patient_id} showed up during these assessment timestamps\n {timestamps}')
    

# Medications Dataframe
The structure is similar to the Steroid dataframe. 

In [ ]:
# first ,last, and random five of medications sheet
display(df_meds.head())
display(df_meds.tail())
display(df_meds.sample(5))

In [ ]:
med_patient = df_meds[df_meds['Digest'] == patient_digest]
display(med_patient)
display(med_patient['Assessment'].drop_duplicates().sort_values())

# Overlap and Confounder Check
As mentioned before we check if and when medications and steroids overlapped. Since all medications start at the 3rd month we can just check the timestamps
We also do a simple Confounder-Check since the steroid injections can potentially lower the DAS28 Score

In [ ]:
#copy original dfs to manipulate safely
med_patient = med_patient.copy()
steroid_patient = steroid_patient.copy()
#preprocessing up datetimes so they are normalized
med_patient['Date of Assessment'] = pd.to_datetime(med_patient['Date of Assessment'], errors='coerce')
steroid_patient['Date of Assessment'] = pd.to_datetime(steroid_patient['Date of Assessment'], errors='coerce')

# sort dates
med_dates = med_patient['Date of Assessment'].drop_duplicates().sort_values().values
steroid_dates = steroid_patient['Date of Assessment'].drop_duplicates().sort_values().values

# look for overlaps
overlap = [d for d in med_dates if d in steroid_dates]

print(f'Med visits: {len(med_dates)} dates')
print(f'Steroid visits: {len(steroid_dates)} dates')
print(f'Overlapping dates: {len(overlap)}')
# check when overlap happened
if len(overlap) > 0:
    for date in overlap:
        formatted = pd.to_datetime(date).strftime('%d %B %Y')
        
        # checks assessment timestamp to know if it happend early or late
        med_assessment = med_patient[med_patient['Date of Assessment'] == date]['Assessment'].values[0]
        steroid_assessment = steroid_patient[steroid_patient['Date of Assessment'] == date]['Assessment'].values[0]
        
        # check if its a first injection
        is_first = steroid_assessment == steroid_patient['Assessment'].min()
        
        print(f'  Date: {formatted}')
        print(f'  Med assessment: Month {med_assessment}')
        print(f'  Steroid assessment: Month {steroid_assessment}')
        print(f'  First injection: {"Yes" if is_first else "No"}')
        
        if steroid_assessment <= 3:
            print(f'First injection')
        else:
            print(f'Later injection')

In [ ]:
expected_months = [0, 3, 6, 9, 12, 15, 18]

# In which months did the patient get steroids and meds 
steroid_months = steroid_patient['Assessment'].drop_duplicates().sort_values().values
med_months = med_patient['Assessment'].drop_duplicates().sort_values().values
# check missing months
missing_steroid_months = [m for m in expected_months if m not in steroid_months]
missing_med_months = [m for m in expected_months if m not in med_months]

if len(missing_med_months) == 0 and len(missing_steroid_months) == 0:
    print(f'Patient was present at all timestamps')
else:
    print(f'Patient was present at: {med_months} for meds and {steroid_months} for steroid injections')
    print(f'Missing meds timestamps: {missing_med_months}')
    print(f'Missing steroid timestamps: {missing_steroid_months}')